# 11 — Donchian (notebook 08 dual channel) × Carver

**Scope:** Only the **first Donchian research notebook** logic — **55/20 prior-bar channels**, breakout / lower-band (+ ATR stop) — blended with the **Carver engine** (`carver.py`). No SSRN 09 Combo, no cross-sectional rank book, no AVWAP/ranked weekly book in this file.

| Book | Meaning |
|------|---------|
| `donchian_nb08` | Daily Donchian 08 weights, equal-weight across universe |
| `carver` | Full Carver forecast weights per name |
| `blend_50_50` | Per-name 50/50 Donchian + Carver |
| `gate` | Carver × 1{Donchian weight > 1%} |

Universe: `DEFAULT_UNIVERSE` (~11 liquid alts + BTC in panel). Costs 10 bps, `shift(1)`. OOS from `SPLIT.oos_start`.

CLI: `python -m research.trend_lab.run_donchian_nb08_carver`


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "research").exists():
    pass
elif (ROOT / "python" / "research").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root", ROOT)


In [ ]:
from __future__ import annotations

import json
from datetime import date
from pathlib import Path

import pandas as pd

from research.trend_lab.allocation import blend_weights
from research.trend_lab.carver_book import carver_weight_panel
from research.trend_lab.data import DEFAULT_UNIVERSE, load_symbol
from research.trend_lab.donchian_combo import COST_BPS, EXEC_LAG, equal_weight_portfolio
from research.trend_lab.donchian_nb08 import Donchian08Params, donchian_nb08_weight_panel
from research.trend_lab.metrics import kpis_from_net
from research.trend_lab.protocol import SPLIT, WARMUP_BARS

START_CAP = 100_000.0
cut = pd.Timestamp(SPLIT.oos_start, tz="UTC")
p = Donchian08Params()


In [ ]:
symbols = ["BTCUSDT"] + [s for s in DEFAULT_UNIVERSE if s != "BTCUSDT"]
ohlcv = {}
for sym in symbols:
    df, src = load_symbol(sym, "1d", start=date(2015, 1, 1))
    if len(df) >= WARMUP_BARS:
        ohlcv[sym] = df
        print(sym, len(df), src)
panel = pd.DataFrame({s: df["close"] for s, df in ohlcv.items()}).sort_index()


In [ ]:
w_don = donchian_nb08_weight_panel(ohlcv, p).reindex(panel.index).fillna(0.0)
w_car = carver_weight_panel(panel, use_cs=True, ann_days=365)
w_blend = pd.DataFrame({s: blend_weights(w_car[s], w_don[s], mix=0.5) for s in panel.columns}, index=panel.index)
w_gate = w_car * (w_don > 0.01).astype(float)

books = {
    "donchian_nb08_eq": equal_weight_portfolio(w_don, panel),
    "carver_eq": equal_weight_portfolio(w_car, panel),
    "blend_50_50_eq": equal_weight_portfolio(w_blend, panel),
    "gate_carver_if_donchian_eq": equal_weight_portfolio(w_gate, panel),
}

def row(name, net):
    oos = net.loc[cut:].fillna(0.0)
    k = kpis_from_net(oos)
    return {"book": name, "oos_sharpe": k["sharpe"], "oos_cagr": k["cagr"],
            "oos_max_dd": k["max_dd"], "oos_pnl_100k": float(START_CAP * ((1 + oos).prod() - 1))}

tbl = pd.DataFrame([row(n, s) for n, s in books.items()]).sort_values("oos_sharpe", ascending=False)
display(tbl.round(4))


In [ ]:
out = Path("/opt/cursor/artifacts/donchian_nb08_carver_results.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({"params": p.__dict__, "results": tbl.to_dict(orient="records")}, indent=2, default=float))
print("wrote", out)


## Donchian only @ ~10% max DD (prop dial)

**IS-only** search on ``target_vol_ann`` (2023 split held out). Channels unchanged (55/20). Not Carver.


In [ ]:
from research.trend_lab.donchian_nb08 import dial_target_vol_for_dd

is_end = cut - pd.Timedelta(days=1)
vt_dd, net_dd = dial_target_vol_for_dd(ohlcv, panel, is_end=is_end, target_dd=-0.10)
p_dd = Donchian08Params(target_vol_ann=vt_dd)
k_is = kpis_from_net(net_dd.loc[:is_end])
k_oos = kpis_from_net(net_dd.loc[cut:])
print("chosen target_vol_ann (IS dial)", vt_dd)
display(pd.DataFrame({"IS_10pct_dial": k_is, "OOS": k_oos}).T.round(4))
print("OOS PnL $100k", round(START_CAP * ((1 + net_dd.loc[cut:].fillna(0)).prod() - 1), 0))
